## **Reinforcement Learning Programming-Assignment 1**

#### **Student name**:Chao-Chung ,Liu
#### **Student ID**: 9067679
#### Link : https://github.com/caatat741213/RLP_Assignment1.git
---

### Problem 1 [10]
* **Pick-and-Place Robot**: Consider using reinforcement learning to control the motion of a robot arm in a repetitive pick-and-place task. If we want to learn movements that are fast and smooth, the learning agent will have to control the motors directly and obtain feedback about the current positions and velocities of the mechanical linkages. Design the reinforcement learning problem as an MDP, define states, actions, rewards with reasoning.

###  🌞 Answer 1

**Agent**:
The robotic arm controller that decides motor actions.

**Environment**:
The robotic arm, object, target location, and workspace.

The agent interacts with the environment by observing states, taking actions, and receiving rewards.

MDP =(S,A,P,R,γ)

| Element | Meaning                |
| ------- | ---------------------- |
| S       | State                  |
| A       | Action                 |
| P       | Transition Probability |
| R       | Reward                 |
| γ       | Discount Factor        |

---

* **Robot Arm: Pick Object -> Move Object -> Place Object**

* **State** :
    * The robotic arm has **two** joints
    * **Joint Position** : Joint1 = 30°,Joint2 = 45°... 
    * **Joint Velocity** : Joint1 speed = 0.5 rad/s, Joint2 speed = 0.4 rad/s ...
    * **Gripper Status** : Open, Closed
    * **Object position** : x,y
    * **Goal position**  : x,y

    S=(θ1​,θ2​,v1​,v2​,g,xobj,yobj,xgoal,ygoal)

    * θ = joint position, θ1, θ2 $\in [0, 2\pi]$
    * v = joint velocity, v1, v2 $\in [-v_{max}, v_{max}]$
    * g = gripper state, g $\in \{0, 1\}$ ,0:Open,1:Closed
    * xobj,yobj = Object position
    * xgoal,ygoal = Goal position

* **Action** :

    Actions directly control the robot motors.Examples include:
    * Increase joint velocity
    * Decrease joint velocity
    * Keep joint velocity
    * Open gripper
    * Close gripper
    * Keep gripper

    $A = \{a_1, a_2, a_g\}$

    * $a_1, a_2 \in \{-1, 0, 1\}$ , -1:speed down , 0:Keep , 1:speed up 
    * $a_g \in \{Open, Keep, Close\}$


* **Transition Probability** :

    $P(s_{t+1} | s_t, a_t)$ , For simplicity, the transition model is assumed to be deterministic.The current state $s_t$ and action $a_t$, the next state $s_{t+1}$ is uniquely determined ,is computed as follows:

    1. Velocity Update: $v_{i, t+1} = v_{i, t} + a_{i, t}$
    2. Position Update: $\theta_{i, t+1} = \theta_{i, t} + v_{i, t+1} \cdot \Delta t$
    3. Gripper State: $g_{t+1}=f(g_t,a_g)$,
        - If $a_g$ = Open  → g=0;
        - If $a_g$ = Close → g=1
        - If $a_g$ = Keep  → g=$g_t$

* **Reward**

    we want to learn movements that are successful,fast and smooth.

    $R(s, a, s') = R_{\text{task}} + R_{\text{time}} + R_{\text{smooth}} + R_{\text{safety}}$

    | Component | Value                     | Purpose |
    | ----------| ------------------------- | ------ |
    | $R_{\text{task}}$ | +100 if success, else 0 | Successfully place object |
    | $R_{\text{time}}$ | -1 per time step | Each time step   |
    | $R_{\text{smooth}}$ | $-5 \sum_{i=1}^2 (\Delta v_i)^2$            | Penalize jerky movements     |
    | $R_{\text{safety}}$ | -20 if collisiont            | Discourage unsafe actions     |

    * Terminal State : An episode terminates when:
        - the object is successfully placed at the goal location
        - a collision occurs

* **Discount Factor**  :
    * γ = 0.99
    * I choose a high discount factor (γ = 0.99) because I want the robot to prioritize reaching the target and successfully completing the task. By doing so, the agent learns to favor the long-term +100 reward rather than trying to minimize the cumulative -1 time penalties by idling or moving aimlessly.



### 🔮Talking Point(Problem 1)

**Talking Point A : Key Reinforcement Learning Feature**

* We represent the robot arm's `joint positions`, `velocities`, `gripper status`, `object position`, and `goal position` as the State $s$ and motor controls as the Action $A$. By defining the transition probability $P(s'|s,a)$ and a reward $R$, we transform a machinery control task into that the agent can solve using RL algorithms.

**Talking Point B : Implementation and Testing Challenge**
* A major challenge is reward shaping.We need to balance the +100 goal reward with the -5 penalty for jerky movements ($R_{\text{smooth}}$). If the smoothness penalty is too high, the robot may move too slowly; if it is too low, the robot may shake during movement.

**Talking Point C : Why This Is Reinforcement Learning**
* Because the robot learns by **Trial and Error** through the agent-environment loop, not by learning from labeled datasets (like in Supervised Learning).Mapping to Sutton & Barto (2018): Our implementation follows the Agent-Environment Interaction cycle described in Chapter 3. The agent observes state $S_t$, takes action $A_t$, and receives a reward $R_{t+1}$ and next state $S_{t+1}$. It uses the Bellman Equation (Section 3.5) to iteratively update its value function.

---

### Problem 2 [20]
**Problem Statement**

**2x2 Gridworld**: Consider a 2x2 gridworld with the following characteristics:
* State Space (S): s1, s2, s3, s4.
* Action Space (A): up, down, left, right.
* Initial Policy (π): For all states, π(up|s) = 1.
* Transition Probabilities P(s′|s, a):
    * If the action is valid (does not run into a wall), the transition is deterministic.
    * Otherwise, s′ = s.
* Rewards R(s):
    * R(s1) = 5 for all actions a.
    * R(s2) = 10 for all actions a.
    * R(s3) = 1 for all actions a.
    * R(s4) = 2 for all actions a.

![Figure1-2x2 Gridworld](images/Figure1-2x2Gridworld.png)

**Tasks**

Perform two iterations of Value Iteration for this gridworld environment. Show the step-by-step process(without code) including policy evaluation and policy improvement. Provide the following for each iteration:
* Iteration 1:
    1. Show the initial value function (V) for each state.
    2. Perform value function updates.
    3. Show the updated value function.
* Iteration 2: Show the value function (V) after the second iteration.

###  🌞 Answer 2

**Grid Layout**
```
| s1(R=5) | s2(R=10)|
| s3(R=1) | s4(R=2) |
```
* Actions are **up**, **down**, **left**, and **right**. Invalid actions keep the agent in the **same state**.

- **States**:$S$: $\{s_1, s_2, s_3, s_4\}$
- **Actions**:$A$: $\{up, down, left, right\}$
- **Initial policy**: $\pi_0$: $\pi(\text{up} \mid s) = 1$ for all $s$
- **Rewards**:$R(s)$: $R(s_1)=5, R(s_2)=10, R(s_3)=1, R(s_4)=2$
- **Discount factor**:$\gamma$ : 0.9(assumed)
- **Value Iteration**: $$V_{k+1}(s) \leftarrow \max_{a} \sum_{s', r} p(s', r | s, a) [r + \gamma V_k(s')]$$

* **Initialization**
    - $V_0(s_1) = V_0(s_2) = V_0(s_3) = V_0(s_4) = 0$

* **First Iteration ($k=1$)**
    * Because the initial value is 0.
    - $V_1(s_1) = \max_a [R(s_1, a) + 0.9 \cdot V_0(s')] = 5 + 0 = 5$
    - $V_1(s_2) = \max_a [R(s_2, a) + 0.9 \cdot V_0(s')] = 10 + 0 = 10$
    - $V_1(s_3) = \max_a [R(s_3, a) + 0.9 \cdot V_0(s')] = 1 + 0 = 1$
    - $V_1(s_4) = \max_a [R(s_4, a) + 0.9 \cdot V_0(s')] = 2 + 0 = 2$

    so:
    * $\gamma = 0.9$，$R(s_1)=5, R(s_2)=10, R(s_3)=1, R(s_4)=2$
    * ($k=1$): $V_1(s_1)=5, V_1(s_2)=10, V_1(s_3)=1, V_1(s_4)=2$

```
| 5 | 10|
| 1 | 2 |
```

* **Second Iteration ($k=2$)**
    * We use the result of $V_1$ to calculate $V_2$
    * $V_2(s) = \max_a [R(s) + 0.9 \cdot V_1(s')]$

    **State s1 ($R=5$):**

    * **Action up (wall)**: $5 + 0.9 \times V_1(s1) = 5 + 0.9 \times 5 = 9.5$
    * **Action down**: $5 + 0.9 \times V_1(s3) = 5 + 0.9 \times 1 = 5.9$
    * **Action left (wall)**: $5 + 0.9 \times V_1(s1) = 5 + 0.9 \times 5 = 9.5$
    * **Action right**: $5 + 0.9 \times V_1(s2) = 5 + 0.9 \times 10 = 14$
    * **result**: $V_2(s1) = \max(9.5,\ 5.9,\ 9.5,\ 14) = 14$ (Best `right`)

    **State s2 ($R=10$):**

    * **Action up (wall)**: $10 + 0.9 \times V_1(s2) = 10 + 0.9 \times 10 = 19$
    * **Action down**: $10 + 0.9 \times V_1(s4) = 10 + 0.9 \times 2 = 11.8$
    * **Action left**: $10 + 0.9 \times V_1(s1) = 10 + 0.9 \times 5 = 14.5$
    * **Action right (wall)**: $10 + 0.9 \times V_1(s2) = 10 + 0.9 \times 10 = 19$
    * **result**: $V_2(s2) = \max(19,\ 11.8,\ 14.5,\ 19) = 19$ (Best `up` or `right`)

    **State s3 ($R=1$):**

    * **Action up**: $1 + 0.9 \times V_1(s1) = 1 + 0.9 \times 5 = 5.5$
    * **Action down (wall)**: $1 + 0.9 \times V_1(s3) = 1 + 0.9 \times 1 = 1.9$
    * **Action left (wall)**: $1 + 0.9 \times V_1(s3) = 1 + 0.9 \times 1 = 1.9$
    * **Action right**: $1 + 0.9 \times V_1(s4) = 1 + 0.9 \times 2 = 2.8$
    * **result**: $V_2(s3) = \max(5.5,\ 1.9,\ 1.9,\ 2.8) = 5.5$ (Best `up`)

    **State s4 ($R=2$):**

    * **Action up**: $2 + 0.9 \times V_1(s2) = 2 + 0.9 \times 10 = 11$
    * **Action down (wall)**: $2 + 0.9 \times V_1(s4) = 2 + 0.9 \times 2 = 3.8$
    * **Action left**: $2 + 0.9 \times V_1(s3) = 2 + 0.9 \times 1 = 2.9$
    * **Action right (wall)**: $2 + 0.9 \times V_1(s4) = 2 + 0.9 \times 2 = 3.8$
    * **result**: $V_2(s4) = \max(11,\ 3.8,\ 2.9,\ 3.8) = 11$ (Best `up`)

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('.')

from src.utils import setup_logger
from src.environments import GridWorld2x2
from src.agents import ValueIterationAgent

# Logger
logger = setup_logger("problem2_execution.log")

# env and agent
env = GridWorld2x2()
agent = ValueIterationAgent(environment=env, gamma=0.9, logger=logger)

# two iterations
final_values = agent.iterate(iterations=2)

print("\nFinal Values:")
for state, value in final_values.items():
    print(f"V({state}) = {value}")

2026-06-11 22:54:13,309 - INFO - Starting Value Iteration for 2 iterations.
2026-06-11 22:54:13,310 - INFO - Iteration 1 Values: {'s1': 5.0, 's2': 10.0, 's3': 1.0, 's4': 2.0}
2026-06-11 22:54:13,310 - INFO - Iteration 2 Values: {'s1': 14.0, 's2': 19.0, 's3': 5.5, 's4': 11.0}


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

Final Values:
V(s1) = 14.0
V(s2) = 19.0
V(s3) = 5.5
V(s4) = 11.0


### 🔮Talking Point(Problem 2)

**Talking Point A : Key Reinforcement Learning Feature**

* 12

**Talking Point B : Implementation and Testing Challenge**
* 12

**Talking Point C : Why This Is Reinforcement Learning**
* 12

---

### Problem 3 [35]

**Problem Statement**

**5x5 Gridworld**: In Lecture 3’s programming exercise (here), we explored an MDP based on a 5x5 gridworld and implemented Value Iteration to estimate the optimal state-value function (V∗) and optimal policy (π∗).

The environment can be described as follows:

![Figure2- 5x5 Gridworld](images/Figure2-5x5Gridworld.png)


* States: states are identified by their row and column, the same as a regular matrix. Ex: the state in row 0 and column 3 is s0,3 (Figure: 2)
    * Terminal/Goal state: The episode ends if the agent reached this state. sGoal = s4,4
    * Grey states: {s2,2, s3,0, s0,4}, these are valid but non-favourable states, as will be seen in the reward function.
* Actions: a1 = right, a2 = down, a3 = down, a4 = up for all states.
* Transitions: If an action is valid, the transition is deterministic, otherwise s′ = s
* Rewards R(s):

**Tasks**

**Task1: Update MDP Code**

1. Update the reward function to be a list of reward based on whether the state is terminal, grey, or a regular state.
2. Run the existing code developed in class and obtain the optimal state-values and optimal policy.Provide a figures of the gridworld with the obtained V∗ and π∗ (You can manually create a table).

**Task 2: Value Iteration Variations**

Implement the following variation of value iteration. Confirm that it reaches the same optimal statevalue function and policy.

1. In-Place Value Iteration: Use a single array to store the state values. This means that you update the value of a state and immediately use that updated value in the subsequent updates.

**Deliverables**

* Full code with comments to explain key steps and calculations.
* Provide the estimated value function for each state.
* Important: Compare the performance of these variations in terms of optimization time, number of episodes, and provide comments on their computational complexity.

###  🌞 Answer 3

### 🔮Talking Point(Problem 3)

**Talking Point A : Key Reinforcement Learning Feature**

* 12

**Talking Point B : Implementation and Testing Challenge**
* 12

**Talking Point C : Why This Is Reinforcement Learning**
* 12

---

### Problem 4 [35]

**Problem Statement**

**Off-policy Monte Carlo with Importance Sampling**: We will use the same environment, states,actions, and rewards in Problem 3.

**Task**

Implement the off-policy Monte Carlo with Importance sampling algorithm to estimate the value function for the given gridworld. Use a fixed behavior policy b(a|s) (e.g., a random policy) to generate episodes and a greedy target policy.

**Suggested steps**

1. Generate multiple episodes using the behavior policy b(a|s).
2. For each episode, calculate the returns (sum of discounted rewards) for each state.
3. Use importance sampling to estimate the value function and update the target policy π(a|s).
4. You can assume a specific discount factor (e.g., γ = 0.9) for this problem.
5. Use the same main algorithm implemented in lecture 4 in class.

**Deliverables**

* Full code with comments to explain key steps and calculations.
* Provide the estimated value function for each state.
* Important Compare the estimated value function obtained from Monte Carlo with the one
obtained from Value Iteration in terms of optimization time, number of episodes, computational complexity, and any other aspects you notice.

###  🌞 Answer 4

### 🔮Talking Point(Problem 4)

**Talking Point A : Key Reinforcement Learning Feature**

* 12

**Talking Point B : Implementation and Testing Challenge**
* 12

**Talking Point C : Why This Is Reinforcement Learning**
* 12